# Gao et al. (2026) on dandiset 001683: band power, asymmetry index and seizure PLI

Paper. Gao H, Frake A, Durand DM, He B (2026). Transcranial focused ultrasound stimulation
targeting white matter inhibits seizures in a rat model of epilepsy. *IEEE Trans Neural Syst Rehabil
Eng* 34, 251-259. [doi:10.1109/tnsre.2025.3644273](https://doi.org/10.1109/tnsre.2025.3644273). The
claims file was written from the NIH author manuscript (PMC12848950) and the supplement.

Dandiset. [DANDI:001683](https://dandiarchive.org/dandiset/001683/0.260109.2104), version
0.260109.2104, the only published version (the paper cites none; see `seizure_metrics.ipynb`).

This notebook streams all 70 labelled EEG blocks (2.9 GB each, about 205 GB) once and reduces each
as it loads (`tfus.process_block`): the Welch PSD of the two labelling channels (8 and 25, Fig. S1)
after a common average reference and a 60 Hz notch, over the whole block and over the seizure
periods; 30 to 80 Hz power in 1 s bins for the clock check of the labels; and the phase lag index
over the seizure periods for every pair of one ipsilateral and one contralateral channel. It takes
about 22 minutes with four processes. It also reads the 13 EEG recordings made before the first
4-AP injection (`EEG data for PLV analysis`) for Figs. S5 and S7.

| Claim | Tested here |
|---|---|
| fig4-no-theta-alpha-gamma-power-change | yes |
| fig4c-beta-decrease | yes |
| figs4-sham-power-no-change | yes |
| fig4ef-ai-increase | yes |
| figs5-seizures-lower-ai | yes, the pre-4-AP state from the PLV recordings (13 of 16 rats) |
| fig6-epileptic-pli-s1-decrease | yes |
| figs7-seizure-pli-s1-vs-non-s1 | yes, as figs5-seizures-lower-ai |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import tfus

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)

started = time.time()
assets = tfus.list_assets()
big = assets[assets["size"] > 2e9]
lab = [m for m in tfus.read_all_meta(big.asset_id.tolist()) if m["kind"] == "Labeled_data"]
M = {m["asset_id"]: m for m in lab}
blocks = tfus.assign_blocks(lab)
print(f"dandiset {tfus.DANDISET_ID} version {tfus.DANDISET_VERSION}: {len(lab)} labelled blocks, "
      f"{sum(m['n_samples'] * m['n_channels'] * 8 for m in lab) / 1e9:.0f} GB of samples")

In [ ]:
t0 = time.time()
R = tfus.process_blocks(lab, workers=4)
print(f"processed {len(R)} blocks in {(time.time() - t0) / 60:.1f} min")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Rats per group:", blocks.groupby("group").subject.nunique().to_dict(), "; blocks per group:", blocks.groupby("group").size().to_dict())
print("Blocks per rat:", blocks.groupby("subject").size().to_dict())
m0 = lab[0]
e = m0["electrodes"]
print("\nElectrode table of the EEG files (x, y in mm; location):")
print(e.assign(channel=np.arange(1, 33)).set_index("channel").T.to_string())
same = all(m["electrodes"][["x", "y"]].equals(m0["electrodes"][["x", "y"]]) for m in lab)
print("Same electrode table in every block file:", same)
print(f"Channel {tfus.IPSI_CH} at x={e.x[tfus.IPSI_CH - 1]}, y={e.y[tfus.IPSI_CH - 1]}; channel {tfus.CONTRA_CH} at x={e.x[tfus.CONTRA_CH - 1]}, y={e.y[tfus.CONTRA_CH - 1]}")
print("Burr holes (Methods A): AP -2 mm, ML +2.5 (right, injection, ipsilateral) and -2.5 (left)")
print(f"Channels 1-16 have x > 0: {bool((e.x[:16] > 0).all())}; channels 17-32 x < 0: {bool((e.x[16:] < 0).all())}")
plv = [m for m in tfus.read_all_meta(assets[(assets['size'] > 0.2e9) & (assets['size'] < 0.4e9)].asset_id.tolist()) if m["kind"].startswith("EEG data for PLV")]
print("\nNotes of the dandiset's PLV files (the source of the S1 channel lists):", sorted({m['notes'] for m in plv}))
f, p = R[m0["asset_id"]]["psd_block"]
print(f"\nPSD: {len(f)} bins, {f[1] - f[0]:.2f} Hz apart, 0 to {f[-1]:.0f} Hz; values {p.min():.2e} to {p.max():.2e} V^2/Hz")
raw = tfus.read_rows(m0, 1_000_000, 1_003_052)
print(f"Raw EEG, 1 s of block 1: {raw.min() * 1e6:.0f} to {raw.max() * 1e6:.0f} uV; per-channel SD {raw.std(axis=0).min() * 1e6:.0f} to {raw.std(axis=0).max() * 1e6:.0f} uV")

The electrode table puts channels 1 to 16 on the right (ipsilateral to the injection) and 17 to 32
on the left, and channels 8 and 25 at ML +/-2.8 mm, AP -3 mm, next to the burr holes, which fits
the encircled electrodes of Fig. S1. The values are in volts, of the order of microvolts (per-channel SD of a few
microvolts in the excerpt), a plausible scale for a skull EEG.

Clock check of the labels. The 30 to 80 Hz power of channel 8 (ipsilateral labels) and
channel 25 (contralateral labels) in 1 s bins, inside the labelled seizures against outside, with
the label times read as minutes, shifted later by 30 s to 600 s, and read as seconds.

In [ ]:
rows = []
for a, r in R.items():
    g = r["gamma_1s"]; nb = len(g)
    for k, h in enumerate(("ipsilateral", "contralateral")):
        s = tfus.seizures(M[a], h)
        for shift in (0, 30, 60, 120, 600, "read as seconds"):
            inside = np.zeros(nb, bool)
            pairs = s[["start", "stop"]].to_numpy() / (60 if shift == "read as seconds" else 1)
            for x, y in pairs:
                off = 0 if shift == "read as seconds" else shift
                inside[max(0, int(x + off)):min(nb, int(np.ceil(y + off)))] = True
            if inside.any() and (~inside).any():
                rows.append({"block": a, "hemi": h, "labels": shift, "inside/outside": g[inside, k].mean() / g[~inside, k].mean()})
cc = pd.DataFrame(rows)
print(cc.groupby("labels", sort=False)["inside/outside"].describe().round(2).to_string())
print(f"\nFile-hemispheres where gamma power is higher inside the labels (as minutes, no shift): "
      f"{(cc[cc.labels == 0]['inside/outside'] > 1).sum()} of {(cc.labels == 0).sum()}")

With the labels read as minutes, 30 to 80 Hz power inside them is many times that outside in every
block and hemisphere (the paper's labelling rule, Methods C); shifted by 60 s or more, or read as
seconds, the contrast is gone. So the labels and the EEG share a clock, and the label times are in
minutes. The two blocks without seizures (BH634 and BH638, post 2) have no labels to test.

In [ ]:
# Per rat, hemisphere, block: band power ratios (M5), under the readings segment (block, seizure)
# and total (3-300 Hz, 0 Hz to Nyquist); and the asymmetry index (Eq. 1) on the band power.
info = blocks.set_index("asset_id")
bp, ai = [], []
for a, r in R.items():
    i = info.loc[a]
    base = {"subject": i.subject, "group": i.group, "block": i.block}
    for seg, key in (("block", None), ("seizure", "seiz")):
        for k, h in enumerate(("ipsilateral", "contralateral")):
            if key is None:
                f, p = r["psd_block"]; p = p[:, k]
            else:
                f, p = r["psd_seiz_ipsi" if k == 0 else "psd_seiz_contra"]
                if f is None:
                    continue
                p = p[:, 0]
            pw = tfus.band_powers(f, p)
            for band in tfus.BANDS:
                for tot in ("total_3_300", "total_all"):
                    bp.append({**base, "hemi": h, "segment": seg, "total": tot, "band": band, "ratio": pw[band] / pw[tot]})
        f, p = r["psd_block"] if key is None else r["psd_seiz_merged"]
        if f is None:
            continue
        pw = tfus.band_powers(f, p)
        for band in tfus.BANDS:
            ip, co = pw[band]
            ai.append({**base, "segment": seg, "band": band, "ai": (ip - co) / (ip + co),
                       "ai_ratio": (ip / pw["total_3_300"][0] - co / pw["total_3_300"][1]) / (ip / pw["total_3_300"][0] + co / pw["total_3_300"][1])})
bp, ai = pd.DataFrame(bp), pd.DataFrame(ai)
print(f"Band power rows: {len(bp)}; AI rows: {len(ai)}")
print("Ranges of the band ratios:", bp.groupby(["band"]).ratio.agg(["min", "max"]).round(4).to_dict("index"))
print("Ranges of AI:", ai.groupby("band").ai.agg(["min", "max"]).round(3).to_dict("index"))
print("Blocks without a seizure PSD (no seizure of 2 s or more):", sorted(set((r.subject, r.block) for r in blocks.itertuples()) - set(map(tuple, bp[bp.segment == 'seizure'][["subject", "block"]].drop_duplicates().to_numpy()))))

## fig4-no-theta-alpha-gamma-power-change

> There are no significant changes in theta, alpha or gamma power ratios in either hemisphere.

A claim of absence. Band power ratios (M5) for sham and tFUS, blocks baseline to post 2, group by
block two-way ANOVA per hemisphere and band, with Bonferroni post hoc tests (M3). The claim covers
the within-group comparisons against baseline. Every reading of M5 (segment: whole block or
seizure periods; total: 3 to 300 Hz or 0 Hz to Nyquist) and of M3 is run. As in
`seizure_metrics.ipynb`, the repeated-measures reading is the one adopted where the ANOVA readings
disagree (see the summary there: it is the reading that gives the paper's own significant
within-group changes of Fig. 3).

In [ ]:
def band_tests(df, value="ratio", bands=tfus.BANDS, by_hemi=True):
    rows = []
    for (seg, tot), d in df.groupby(["segment", "total"]):
        for band in bands:
            for h in (("ipsilateral", "contralateral") if by_hemi else (None,)):
                dd = d[(d.band == band) & ((d.hemi == h) if h else True)]
                for k, (table, post) in tfus.group_block_tests(dd, value).items():
                    for r in post.itertuples():
                        rows.append({"segment": seg, "total": tot, "band": band, "hemi": h, "anova": k,
                                     "comparison": r.comparison, "family": r.family, "diff": r.diff,
                                     "p_family": r.p_family, "p_all": r.p_all})
    return pd.DataFrame(rows)

T4 = band_tests(bp)
w = T4[T4.family == "within"]
summary = (w.assign(sig_i=w.p_family < 0.05, sig_ii=w.p_all < 0.05)
             .groupby(["band", "segment", "total", "anova"])[["sig_i", "sig_ii"]].sum())
print("Number of significant within-group comparisons against baseline (of 12: 2 groups x 3 blocks x 2 hemispheres):")
print(summary.unstack("anova").to_string())
print("\nThe significant ones, theta, alpha and gamma:")
print(w[(w.band != "beta") & (w.p_family < 0.05)].round(4).to_string(index=False))

In [ ]:
print("Group mean band ratios (seizure segment, 3-300 Hz total):")
print(bp[(bp.segment == "seizure") & (bp.total == "total_3_300") & (bp.block != "post3")]
      .groupby(["band", "hemi", "group", "block"]).ratio.mean().unstack().round(4).to_string())
print("\nGroup mean band ratios (whole block, 3-300 Hz total):")
print(bp[(bp.segment == "block") & (bp.total == "total_3_300") & (bp.block != "post3")]
      .groupby(["band", "hemi", "group", "block"]).ratio.mean().unstack().round(4).to_string())
print("\nBetween-group comparisons (reported, not deciding), significant under the own family:")
b = T4[(T4.family == "between") & (T4.p_family < 0.05)]
print(b.round(4).to_string(index=False) if len(b) else "none")

**Result: not reproduced, resting on an assumption.** The ANOVA readings disagree, and the M5
readings agree within each. Under the ordinary reading no within-group comparison against baseline
is significant for theta, alpha or gamma under any of the four M5 readings. Under the
repeated-measures reading, adopted as the paper's test (see the summary), the sham theta ratio
rises above baseline in 4 or 5 of the 12 comparisons under every M5 reading (ipsilateral post 2:
Bonferroni p = 0.017 over the whole block, where the 3-300 Hz ratio goes from 0.098 to 0.136, and
p = 0.0018 over the seizure periods, from 0.078 to 0.131). Alpha and gamma have none. No
between-group comparison is significant.

## fig4c-beta-decrease

> The beta power ratio of the tFUS group decreased significantly during stimulation and post 1
> compared with baseline.

Reproduced when, under a combination of the M3, M5 and hemisphere readings, the condition block
and post 1 are both significantly lower than baseline.

In [ ]:
wb = T4[(T4.band == "beta") & T4.comparison.isin(["tFUS baseline-condition", "tFUS baseline-post1"])]
print("tFUS beta ratio, baseline minus block (positive = decrease), with p under families (i) and (ii):")
print(wb.pivot_table(index=["segment", "total", "anova", "hemi"], columns="comparison", values=["diff", "p_family", "p_all"]).round(4).to_string())
both = wb.assign(ok=(wb.p_family < 0.05) & (wb["diff"] > 0)).groupby(["segment", "total", "anova", "hemi"]).ok.sum()
print("\nCombinations with both comparisons significant decreases (family i):", list(both[both == 2].index))
print("\ntFUS beta ratio means:")
print(bp[(bp.band == "beta") & (bp.group == "tFUS") & (bp.block != "post3")].groupby(["segment", "total", "hemi", "block"]).ratio.mean().unstack().round(4).to_string())

**Result: partially reproduced.** Over the whole block under the repeated-measures reading the
tFUS beta ratio at post 1 is significantly below baseline in both hemispheres (3-300 Hz total:
p = 0.0067 ipsilateral, 0.040 contralateral; 0 Hz to Nyquist: 0.0019 and 0.018), but the condition
block is not (p = 0.19 to 0.64), although its mean is lower (ipsilateral 0.445 at baseline, 0.417
at the condition block). Over the seizure periods the beta ratio does not fall (ipsilateral 0.493,
0.487 and 0.500) and no comparison is significant. Under the ordinary reading none is.

## figs4-sham-power-no-change

> In the sham group, band power ratios show no significant change among time blocks or between
> hemispheres.

A claim of absence. Sham rats, five blocks, per band a two-way ANOVA (hemisphere by block) with
Bonferroni post hoc tests; no block or hemisphere effect nor post hoc comparison may be significant
in any band.

In [ ]:
rows = []
for (seg, tot), d in bp[bp.group == "sham"].groupby(["segment", "total"]):
    for band in tfus.BANDS:
        dd = d[d.band == band]
        for k, (table, post) in tfus.hemi_block_tests(dd, "ratio").items():
            rows.append({"segment": seg, "total": tot, "band": band, "anova": k,
                         "p_hemi": table.p.iloc[0], "p_block": table.p.iloc[1], "p_interaction": table.p.iloc[2],
                         "posthoc_sig_i": int((post.p_family < 0.05).sum()), "posthoc_sig_ii": int((post.p_all < 0.05).sum()),
                         "smallest": post.loc[post.p_family.idxmin(), "comparison"], "smallest_p": post.p_family.min()})
S4 = pd.DataFrame(rows)
print(S4.round(4).to_string(index=False))

**Result: not reproduced, resting on an assumption.** Beta varies with block and hemisphere: over
the whole block the block and hemisphere effects are significant under both readings (ordinary
block p = 0.0004 and 0.0002 for the two totals, hemisphere p below 0.0001, with 4 and 5 post hoc
comparisons significant), and over the seizure periods under the repeated-measures reading (block
p = 0.0030 and 0.0025, 1 and 2 post hoc comparisons). Gamma has a hemisphere effect over the whole
block (ordinary p = 0.0037 and 0.0004) and, under the repeated reading, over the seizure periods
too (p = 0.028 and 0.013); the repeated reading also finds a theta block effect (p = 0.011 to
0.040). Under the ordinary reading over the seizure periods nothing is significant in any band
(smallest p = 0.09), so the M5 readings disagree under that ANOVA reading; under the
repeated-measures reading, the one adopted, they agree.

## fig4ef-ai-increase

> The normalized asymmetry index of the tFUS group increases significantly relative to baseline in
> the theta and alpha bands, and differs significantly from the sham group in every band.

AI per rat, band and block (Eq. 1, on the band power of channels 8 and 25), normalized to the
group's mean baseline AI by (a) division or (b) subtraction (M5), for the whole block and for the
seizure periods (the union of both hemispheres' labels); group by block two-way ANOVA per band.
Reproduced when, under one combination of readings, at least one tFUS block is significantly above
baseline in theta and in alpha, and sham against tFUS is significant at one block or more in each
of the four bands.

In [ ]:
# AI on the band power (`ai`) and, as a second reading (verifier, round 1), on the band over 3-300 Hz
# ratios (`ai_ratio`): the paper calls the ratio "absolute power" and Eq. 1 says "PSD".
d = ai[ai.block != "post3"].copy()
rows, bases = [], []
for value in ("ai", "ai_ratio"):
    base = d[d.block == "baseline"].groupby(["segment", "band", "group"])[value].mean().rename("base")
    dv = d.join(base, on=["segment", "band", "group"])
    dv["norm_div"] = dv[value] / dv.base
    dv["norm_sub"] = dv[value] - dv.base
    bases.append(base.rename(value))
    for seg in ("block", "seizure"):
        for norm in ("norm_div", "norm_sub"):
            for band in tfus.BANDS:
                dd = dv[(dv.segment == seg) & (dv.band == band)]
                for k, (table, post) in tfus.group_block_tests(dd, norm).items():
                    for r in post.itertuples():
                        rows.append({"value": value, "segment": seg, "norm": norm, "band": band, "anova": k, "comparison": r.comparison,
                                     "family": r.family, "diff": r.diff, "p_family": r.p_family, "p_all": r.p_all,
                                     "tFUS base": base.loc[(seg, band, "tFUS")]})
    if value == "ai":
        print("Group means of normalized AI (AI on band power):")
        print(dv.groupby(["segment", "band", "group", "block"])[["norm_div", "norm_sub"]].mean().unstack().round(3).to_string())
TA = pd.DataFrame(rows)
print("\nGroup mean baseline AI, on band power and on band ratios (its sign decides what an increase means under division):")
print(pd.concat(bases, axis=1).unstack().round(4).to_string())

In [ ]:
def ai_verdict(t):
    # "increase", two readings: the normalized value rises above its baseline (baseline minus block < 0),
    # or the AI itself rises; they agree under subtraction and are opposite under division when the
    # group's baseline AI is negative (then a rising AI gives a falling ratio).
    up_norm, up_ai = {}, {}
    for band in ("theta", "alpha"):
        x = t[(t.band == band) & t.comparison.str.startswith("tFUS baseline") & (t.p_family < 0.05)]
        up_norm[band] = bool((x["diff"] < 0).any())
        flip = (t.name[2] == "norm_div") & (x["tFUS base"] < 0)
        up_ai[band] = bool((np.where(flip, x["diff"] > 0, x["diff"] < 0)).any())
    between = {band: bool((t[(t.band == band) & (t.family == "between")].p_family < 0.05).any()) for band in tfus.BANDS}
    return pd.Series({**{f"norm up {k}": v for k, v in up_norm.items()}, **{f"AI up {k}": v for k, v in up_ai.items()},
                      **{f"between {k}": v for k, v in between.items()}})

V = TA.groupby(["value", "segment", "norm", "anova"]).apply(ai_verdict)
print(V.to_string())
print("\nSignificant comparisons (own family):")
print(TA[TA.p_family < 0.05].round(4).to_string(index=False))

**Result: partially reproduced, resting on an assumption.** With AI on the band power (Eq. 1), the mean baseline AI is
negative in both groups (whole block: sham -0.58 to -0.61, tFUS -0.24 to -0.29), so the two
normalizations read an increase in opposite directions. Over the whole block under the
repeated-measures reading the tFUS AI rises above baseline, toward balance, in theta and alpha
under both normalizations once the sign is read this way (subtraction: theta post 1 p = 0.0495,
post 2 p = 0.010, alpha post 2 p = 0.012; division, where a rising AI is a falling ratio: theta
post 2 p = 0.0020, alpha post 2 p = 0.0005). Read literally, as the normalized value rising, only
subtraction gives it. Over the seizure periods nothing is significant. Sham against tFUS is not
significant in any band, block or combination of readings. With AI on the band ratios (the
verifier's second reading) the baseline AI is near 0 (-0.09 to +0.04), no tFUS block differs from
baseline in theta or alpha, and no group difference is significant. So only the first part of the
claim holds, under one combination of readings, and the outcome rests on computing AI on the band
power: on band ratios it would be not reproduced.

## figs5-seizures-lower-ai

> During seizures (baseline block, after 4-AP) the asymmetry index is negative and differs from
> zero in every band, more than in the pre-4-AP baseline state; seizures disrupt the normal
> interhemispheric balance.

The claims file reads the pre-4-AP state as (a) the resting baseline recording of the target
confirmation or (b) the first recording before the first injection, where the dataset holds one.
Reading (a) cannot be run: the resting recordings (`EP data`, `BASELINE`) hold only the 64
intracranial channels of left S1, not the EEG. Reading (b) can, for 13 of the 16 rats (all 6 sham
and 7 of the 10 tFUS rats; BH638, BH642 and BH644 have none): each rat's EEG file labelled
`EEG data for PLV analysis` is its last recording before the first labelled block, and it ends 64
to 176 min before that block starts (table below), so before the monitoring block of about
62.5 min that the first injection opens (Fig. 1E, Methods D). The first version of this attempt
missed this and blocked the claim; the verifier found it. That recording carries a 20 ms tFUS
pulse every 2 s, so the pre-4-AP state is taken from the gaps between pulses, 250 ms after one
onset to 20 ms before the next (`tfus.pre_state`), with the whole recording as a variant. This is
an assumption: the paper's pre-4-AP baseline may have been a recording without pulses.

AI (Eq. 1, not normalized) on the band power of channels 8 and 25, and, as a second reading, on
the band over 3-300 Hz ratios (the paper calls that ratio "absolute power"; Eq. 1 says "PSD").
Seizure AI is that of the merged seizure periods of the baseline block. Rats: (a) all 16 (13 with
a pre-4-AP recording), (b) the 10 tFUS rats (7). Tests: one-sample t against 0 per band (Fig. S5
legend); seizure against pre-4-AP by paired t and by a two-way ANOVA (state by band, ordinary and
repeated measures) with Bonferroni post hoc tests per band.

In [ ]:
# The pre-4-AP recordings: timing against each rat's first labelled block, then band power and PLI.
first = blocks[blocks.order == 0].set_index("subject")
rows = []
for m in plv:
    if m["subject"] not in first.index:
        continue
    t0 = pd.Timestamp(m["start"]); t1 = t0 + pd.Timedelta(seconds=m["n_samples"] / m["rate"])
    later = [pd.Timestamp(x["start"]) for x in lab if x["subject"] == m["subject"]]
    rows.append({"subject": m["subject"], "group": first.loc[m["subject"], "group"], "PLV start": str(t0)[:19],
                 "PLV minutes": round((t1 - t0).total_seconds() / 60, 1),
                 "min from PLV end to first block": round((pd.Timestamp(first.loc[m["subject"], "start"]) - t1).total_seconds() / 60, 1),
                 "first block is earliest labelled": pd.Timestamp(first.loc[m["subject"], "start"]) == min(later)})
PT = pd.DataFrame(rows).sort_values(["group", "subject"])
print(PT.to_string(index=False))
print("Epilepsy rats without a PLV recording:", sorted(set(blocks.subject) - set(PT.subject)))
t0 = time.time()
PRE = tfus.pre_states([m for m in plv if m["subject"] in set(PT.subject)], workers=4)
print(f"pre-4-AP state of {len(PRE)} rats in {time.time() - t0:.0f} s; gaps per recording "
      f"{min(r['n_gaps'] for r in PRE.values())} to {max(r['n_gaps'] for r in PRE.values())}, gap length "
      f"{min(r['gap_s'] for r in PRE.values()):.2f} to {max(r['gap_s'] for r in PRE.values()):.2f} s")
rows = []
for s, r in PRE.items():
    for reading in ("gaps", "whole"):
        f, p = r["psd_" + reading]
        pw = tfus.band_powers(f, p)
        for band in tfus.BANDS:
            ip, co = pw[band]
            ri, rc = ip / pw["total_3_300"][0], co / pw["total_3_300"][1]
            rows.append({"subject": s, "group": first.loc[s, "group"], "pre": reading, "band": band,
                         "ai": (ip - co) / (ip + co), "ai_ratio": (ri - rc) / (ri + rc)})
PA = pd.DataFrame(rows)
print("\nPre-4-AP AI, mean (min to max) over the 13 rats:")
print(PA.groupby(["pre", "band"])[["ai", "ai_ratio"]].agg(["mean", "min", "max"]).round(3).to_string())

In [ ]:
sa = ai[(ai.segment == "seizure") & (ai.block == "baseline")]
rows = []
for value in ("ai", "ai_ratio"):
    for rats, dd in (("16 rats", sa), ("10 tFUS rats", sa[sa.group == "tFUS"]),
                     ("13 rats with pre", sa[sa.subject.isin(PRE)]), ("7 tFUS rats with pre", sa[sa.subject.isin(PRE) & (sa.group == "tFUS")])):
        for band in tfus.BANDS:
            x = dd[dd.band == band][value]
            t = stats.ttest_1samp(x, 0)
            rows.append({"AI on": value, "rats": rats, "band": band, "n": len(x), "seizure AI": x.mean(), "sem": x.sem(), "t": t.statistic, "p": t.pvalue})
print("Seizure AI of the baseline block against 0 (one-sample t):")
print(pd.DataFrame(rows).round(4).to_string(index=False))
rows, verdict = [], []
for value in ("ai", "ai_ratio"):
    for pre in ("gaps", "whole"):
        for rats in ("13 rats", "7 tFUS rats"):
            pp = PA[(PA.pre == pre) & ((PA.group == "tFUS") if rats.startswith("7") else True)]
            m = pp[["subject", "band", value]].rename(columns={value: "pre"}).merge(
                sa[["subject", "band", value]].rename(columns={value: "seizure"}), on=["subject", "band"])
            long = pd.concat([m.assign(state="pre", y=m.pre), m.assign(state="seizure", y=m.seizure)])[["subject", "band", "state", "y"]]
            comps = [(band, {"state": "seizure", "band": band}, {"state": "pre", "band": band}, "bands") for band in tfus.BANDS]
            anova = {k: tfus.twoway(long, "y", "state", "band", "subject", comps, rm=(k == "repeated"))[1].set_index("comparison")
                     for k in ("ordinary", "repeated")}
            sig0, lower = {}, {}
            for band in tfus.BANDS:
                x = m[m.band == band]
                t0 = stats.ttest_1samp(x.seizure, 0); tp = stats.ttest_1samp(x.pre, 0); tr = stats.ttest_rel(x.seizure, x.pre)
                rows.append({"AI on": value, "pre": pre, "rats": rats, "band": band, "n": len(x),
                             "seizure": x.seizure.mean(), "p seizure vs 0": t0.pvalue, "pre-4-AP": x.pre.mean(), "p pre vs 0": tp.pvalue,
                             "seizure - pre": (x.seizure - x.pre).mean(), "p paired": tr.pvalue,
                             "p ANOVA ordinary": anova["ordinary"].loc[band, "p_family"], "p ANOVA repeated": anova["repeated"].loc[band, "p_family"]})
                sig0[band] = (t0.pvalue < 0.05) and x.seizure.mean() < 0
                lower[band] = {k: (p < 0.05) and (x.seizure - x.pre).mean() < 0 for k, p in
                               (("paired", tr.pvalue), ("ordinary", anova["ordinary"].loc[band, "p_family"]), ("repeated", anova["repeated"].loc[band, "p_family"]))}
            for test in ("paired", "ordinary", "repeated"):
                verdict.append({"AI on": value, "pre": pre, "rats": rats, "test": test, "seizure below 0 in all bands": all(sig0.values()),
                                "seizure below pre in": ", ".join(b for b in tfus.BANDS if lower[b][test]) or "none",
                                "reproduced": all(sig0.values()) and lower["alpha"][test] and lower["beta"][test]})
S5 = pd.DataFrame(rows)
print("\nSeizure against pre-4-AP AI, per band (p of the ANOVA post hoc tests Bonferroni over the four bands):")
print(S5.round(4).to_string(index=False))
print("\nCriterion per combination of readings:")
print(pd.DataFrame(verdict).to_string(index=False))

**Result: reproduced, resting on assumptions.** With AI on the band power, the seizure AI of the
baseline block is negative and below 0 in every band: -0.41 to -0.43 over the 16 rats (p < 0.0001
in each band), -0.27 to -0.30 over the 10 tFUS rats (p = 0.0001 to 0.0012), and the same over the
13 and 7 rats that have a pre-4-AP recording (p < 0.0001; p = 0.004 to 0.014). It is lower than the
pre-4-AP AI of the gaps between pulses in every band, for all 13 rats (seizure - pre -0.19 to
-0.28; paired t p = 0.0006 or less; repeated-measures ANOVA p < 0.0001) and for the 7 tFUS rats
(paired p = 0.0003 to 0.022; repeated p = 0.018 or less); the whole recording gives the same.
The ordinary two-way ANOVA finds it in theta and alpha only for the 13 rats (p = 0.028 and 0.039;
beta 0.095) and in no band for the 7. The pre-4-AP AI is itself negative (-0.17 to -0.26, below 0
in every band for the 13 rats, p = 0.0007 to 0.045; for the 7 tFUS rats only in gamma, p = 0.022,
the band Fig. S5 marks). The outcome rests on two choices. (1) The pre-4-AP state is the gaps
between the 20 ms pulses of the last recording before the first injection, the only pre-4-AP EEG
in the dandiset. (2) AI is computed on the band power, as Methods E says ("the relative difference
in PSD") and the Fig. 4 legend ("calculated using PSD values"). On the band over 3-300 Hz ratios
the seizure AI is near 0 (-0.04 to +0.01) and not below 0 in any band (p = 0.08 to 0.99), and the
claim fails under every combination.

## fig6-epileptic-pli-s1-decrease

> In epileptic rats, the S1 PLI of the tFUS group decreases in theta, alpha, beta and gamma bands
> during stimulation, post 1 and post 2 relative to baseline seizures, while non-S1 PLI does not
> decrease; the decrease of S1 connectivity is significant.

PLI (M6) over the seizure periods of each block (the union of both hemispheres' labels, periods of
1 s or more), per band, averaged over the 16 S1 pairs (left S1 with right S1 channels) and the 144
non-S1 pairs; per seizure and averaged (reading a) or over the concatenated seizure samples
(reading b); normalized to the group's mean baseline-block PLI by difference or ratio. Reproduced
when, under one combination of readings, the tFUS S1 mean is below baseline in all four bands for
the condition block, post 1 and post 2, the tFUS non-S1 mean is not below baseline in most of
those 12 cells, and a group by block two-way ANOVA per band and region finds sham against tFUS
significant for S1 in at least one band and block.

In [ ]:
rows = []
for a, r in R.items():
    i = info.loc[a]
    for band in tfus.BANDS:
        v = r["pli"][band]
        if v is None:
            continue
        for reading in ("per_seizure", "concat"):
            pm = tfus.pair_means(v[reading])
            for region in ("S1", "nonS1"):
                rows.append({"subject": i.subject, "group": i.group, "block": i.block, "band": band,
                             "reading": reading, "region": region, "pli": pm[region], "n_seizures": v["n"], "seconds": v["seconds"]})
P = pd.DataFrame(rows)
P = P[P.block != "post3"]
print("Rat-blocks with seizure PLI:", P[["subject", "block"]].drop_duplicates().shape[0], "of", int((blocks.block != "post3").sum()))
print("PLI range:", round(P.pli.min(), 3), "to", round(P.pli.max(), 3))
base = P[P.block == "baseline"].groupby(["group", "band", "reading", "region"]).pli.mean().rename("base")
P = P.join(base, on=["group", "band", "reading", "region"])
P["diff"] = P.pli - P.base
P["ratio"] = P.pli / P.base
G = P.groupby(["reading", "region", "band", "group", "block"])[["diff", "ratio"]].mean()
print("\nGroup mean normalized PLI (difference from the group's baseline):")
print(G["diff"].unstack().round(4).to_string())

In [ ]:
rows = []
for reading in ("per_seizure", "concat"):
    for norm in ("diff", "ratio"):
        g = G[norm].xs(reading, level="reading").unstack()
        ref = 0.0 if norm == "diff" else 1.0
        t = g.xs("tFUS", level="group")
        s1 = t.xs("S1", level="region")[["condition", "post1", "post2"]]
        ns = t.xs("nonS1", level="region")[["condition", "post1", "post2"]]
        sig = []
        for band in tfus.BANDS:
            dd = P[(P.reading == reading) & (P.region == "S1") & (P.band == band)]
            for k, (table, post) in tfus.group_block_tests(dd, norm).items():
                for r in post[(post.family == "between") & (post.p_family < 0.05)].itertuples():
                    sig.append(f"{band} {r.comparison} [{k}] p={r.p_family:.3g}")
        rows.append({"reading": reading, "norm": norm, "tFUS S1 cells below baseline (of 12)": int((s1 < ref).sum().sum()),
                     "tFUS non-S1 cells not below baseline (of 12)": int((ns >= ref).sum().sum()),
                     "S1 sham-tFUS significant": "; ".join(sig) if sig else "none"})
print(pd.DataFrame(rows).to_string(index=False))
print("\nPer rat, tFUS S1 PLI difference from the group baseline, per_seizure reading:")
print(P[(P.group == "tFUS") & (P.region == "S1") & (P.reading == "per_seizure")].pivot_table(index=["band", "subject"], columns="block", values="diff").round(3).unstack(0).to_string())

**Result: not reproduced.** Per seizure and averaged (reading a of M6), the tFUS S1 PLI is below
the group's baseline in 8 of the 12 band and block cells: in theta and alpha at all three blocks,
in beta only at the condition block and in gamma only at post 1; over the concatenated seizure
samples, 6 of 12. The non-S1 PLI falls as well: it is not below baseline in only 4 of 12 cells
under every reading, where the claim needs most of the 12. No sham against tFUS comparison of S1
PLI is significant in any band or block under either ANOVA reading. So the S1 decrease holds in two
bands only, the S1 specificity fails and the group difference fails. The first version of this
attempt called this partial, as the criterion's partial clause names the first and last of these
failures; the verifier judged it not reproduced, since the non-S1 part, which the partial clause
does not cover, fails too, and this revision agrees.

## figs7-seizure-pli-s1-vs-non-s1

> Comparing seizures (baseline block) with the pre-4-AP baseline state, the PLI change of S1 pairs
> differs significantly from that of non-S1 pairs in every band.

The pre-4-AP state is the gaps between pulses of each rat's PLV recording, as for
figs5-seizures-lower-ai (13 of the 16 rats; reading (a) of the state cannot be run). Per rat, band
and region, the change is the seizure PLI of the baseline block minus (reading a) or over
(reading b) the pre-4-AP PLI; the seizure PLI per seizure averaged or over the concatenated seizure
samples (M6), the pre-4-AP PLI over the concatenated gap samples. The pre-4-AP PLI per gap,
averaged, is shown as well; its gaps of 1.7 s are far shorter than the seizures, so it is biased
upwards against them (see the PLI bias in `wild_type.ipynb`) and it is not used for the change.
Two-way ANOVA (region by band, ordinary and repeated measures) of the change with Bonferroni post
hoc tests of S1 against non-S1 per band; rats (a) all 13, (b) the 7 tFUS rats. Reproduced when, under
one combination, S1 is significantly higher than non-S1 in all four bands.

In [ ]:
pre_pli = []
for s, r in PRE.items():
    for band in tfus.BANDS:
        for reading in ("concat", "per_gap", "whole"):
            pm = tfus.pair_means(r["pli"][band][reading])
            for region in ("S1", "nonS1"):
                pre_pli.append({"subject": s, "band": band, "pre reading": reading, "region": region, "pre": pm[region]})
pre_pli = pd.DataFrame(pre_pli)
print("Pre-4-AP PLI, mean over the 13 rats (concatenated gaps, per gap, whole recording):")
print(pre_pli.pivot_table(index=["band", "region"], columns="pre reading", values="pre").round(4).to_string())
sz = P[(P.block == "baseline")][["subject", "group", "band", "reading", "region", "pli"]]
C = sz.merge(pre_pli[pre_pli["pre reading"] == "concat"][["subject", "band", "region", "pre"]], on=["subject", "band", "region"])
C["diff"] = C.pli - C.pre
C["ratio"] = C.pli / C.pre
print("\nSeizure PLI of the baseline block, pre-4-AP PLI and the change, mean over the 13 rats:")
print(C.groupby(["reading", "band", "region"])[["pli", "pre", "diff", "ratio"]].mean().round(4).unstack("region").to_string())
rows, verdict = [], []
for reading in ("per_seizure", "concat"):
    for norm in ("diff", "ratio"):
        for rats in ("13 rats", "7 tFUS rats"):
            dd = C[(C.reading == reading) & ((C.group == "tFUS") if rats.startswith("7") else True)]
            comps = [(band, {"region": "S1", "band": band}, {"region": "nonS1", "band": band}, "bands") for band in tfus.BANDS]
            for k in ("ordinary", "repeated"):
                post = tfus.twoway(dd, norm, "region", "band", "subject", comps, rm=(k == "repeated"))[1].set_index("comparison")
                ok = [b for b in tfus.BANDS if post.loc[b, "p_family"] < 0.05 and post.loc[b, "diff"] > 0]
                for b in tfus.BANDS:
                    rows.append({"seizure reading": reading, "change": norm, "rats": rats, "anova": k, "band": b,
                                 "S1 - nonS1": post.loc[b, "diff"], "p": post.loc[b, "p_family"]})
                verdict.append({"seizure reading": reading, "change": norm, "rats": rats, "anova": k,
                                "bands with S1 significantly higher": ", ".join(ok) or "none", "n": len(ok)})
print("\nS1 against non-S1 change per band (Bonferroni over the four bands):")
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("\nCriterion per combination of readings:")
print(pd.DataFrame(verdict).to_string(index=False))

**Result: partially reproduced, resting on an assumption.** With the pre-4-AP state taken from the
gaps between pulses of the PLV recording (13 rats), the seizure PLI of the baseline block is 2 to 5
times the pre-4-AP PLI in both regions (concatenated reading: S1 0.10 to 0.24 against 0.03 to
0.08). The S1 change is larger than the non-S1 change in every band (difference reading, S1 minus
non-S1 0.006 to 0.087), but significantly so only in theta under every ANOVA and rat reading over
concatenated samples (13 rats p = 0.0067 ordinary, 0.0004 repeated; 7 tFUS rats 0.024 and 0.026),
and in theta and gamma per seizure under the repeated reading (p = 0.0017 and 0.044). Alpha and
beta never reach significance, and the ratio reading finds nothing. One or two of four bands is
the criterion's partial outcome. Unlike Fig. S7B, where seizure non-S1 is below zero, both changes
here are positive. The per-gap pre-4-AP PLI (0.09 to 0.31) is 1.7 to 4 times the concatenated
one, the short-window bias, so it is not used.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig4-no-theta-alpha-gamma-power-change | not reproduced (assumed) |
| fig4c-beta-decrease | partially reproduced |
| figs4-sham-power-no-change | not reproduced (assumed) |
| fig4ef-ai-increase | partially reproduced (assumed) |
| figs5-seizures-lower-ai | reproduced (assumed) |
| fig6-epileptic-pli-s1-decrease | not reproduced |
| figs7-seizure-pli-s1-vs-non-s1 | partially reproduced (assumed) |

The outcomes of the other fifteen claims are in `seizure_metrics.ipynb` and `wild_type.ipynb`.

How the analysis differs from the paper's. No ICA is applied: which components the authors
removed is not recorded. The EEG is re-referenced to the common average and notched at 60 Hz, as
in Methods C. The labelling channels 8 and 25 are read from the drawing of Fig. S1 and the S1
channels from the notes of the dandiset's PLV files; the paper's text lists neither. The segment
analysed, the total of the band ratio and the normalization of AI and PLI are not fixed by the
paper and are run under each reading. AI is computed on the band power, as Methods E and the
Fig. 4 legend say, with AI on the band ratios shown as a second reading; the two disagree for
figs5-seizures-lower-ai and fig4ef-ai-increase, which rest on this choice. The repeated-measures ANOVA reading, a mixed model with t and F tests on
Satterthwaite degrees of freedom, is adopted where the ANOVA readings disagree on a claim of
absence, as in `seizure_metrics.ipynb`. The pre-4-AP state of Figs. S5 and S7 is taken from the
gaps between the 20 ms pulses of each rat's last recording before the first injection, which 13 of
the 16 rats have; the paper's own pre-4-AP baseline may have been a recording without pulses.
During the condition block of the tFUS rats the EEG carries the pulse responses at 1 Hz; the paper
does not say whether they were removed, and they are not.

What was found about the dataset. The seizure labels share the EEG's clock and are in minutes:
30 to 80 Hz power inside them is 2.5 to 50 times that outside, and the contrast is gone when they
are shifted by a minute or read as seconds. The EEG before the first 4-AP injection is there for 13
of the 16 rats, as the PLV recordings, which end 64 to 176 min before the baseline block; they are
not described as pre-4-AP recordings and carry tFUS pulses. The baseline AI on the band power is
strongly negative in both groups (mean contralateral band power about 1.7 (tFUS) to 4 (sham) times
the ipsilateral), and negative before 4-AP too (-0.17 to -0.26), while the AI on band ratios is near
0, so the asymmetry is in the total power of the two channels; the paper reports only AI normalized
to the baseline.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")